# Modelo baseline

Passo a Passo para Construção do Modelo
1. Pré-processamento das Imagens
	- Objetivo: Preparar as imagens padronizadas para extração de características.
	- Operações:
		1.	Carregar as imagens padronizadas (256x256) da pasta.
		2.	Normalizar as imagens para reduzir a influência de variações na iluminação.
		3.	Aplicar detecção de bordas usando o operador Sobel.
		4.	Combinar as bordas detectadas com a imagem binária resultante de um threshold lógico (AND lógico).
		5.	Remover ruídos utilizando operações morfológicas.
2. Extração de Características
	- Objetivo: Extrair representações discriminativas das imagens.
	- Métodos sugeridos no artigo:
		1.	PCANet: Extrai componentes principais da imagem, reduzindo a dimensionalidade e mantendo informações relevantes.
		2.	VGGNet: Usa uma rede convolucional pré-treinada para extrair características profundas da imagem (ex.: camada `fc7`).
		3.	Filtro Gabor: Extrai informações texturais e estruturais em diferentes escalas e orientações.
3. Organização dos Dados
	- Objetivo: Organizar os dados em conjuntos de treinamento e teste.
	- Operações:
		1.	Carregar o arquivo parquet contendo os códigos das amostras e suas classes.
		2.	Associar cada imagem padronizada à sua classe com base no código.
		3.	Dividir os dados em conjuntos de treinamento (80%) e teste (20%).
4. Treinamento do Classificador
	- Objetivo: Treinar um modelo SVM para classificar as amostras com base nas características extraídas.
	- Operações:
		1.	Usar as características extraídas como entrada (`X`) e as classes como rótulos (`y`).
		2.	Treinar um SVM com kernel linear ou RBF dependendo da complexidade dos dados.
5. Avaliação do Modelo
	- Objetivo: Avaliar o desempenho do modelo treinado no conjunto de teste.
	- Métrica principal: Acurácia.
	- Comparar os resultados com diferentes métodos de extração de características (PCANet, VGGNet, Gabor).

In [1]:
import pandas as pd
import cv2
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pcanet import PCANet


## Pré-processamento

In [2]:
import json
# Carregar o arquivo parquet
dados = pd.read_parquet('../data/bronze/dados_fardinhos.parquet')

with open('../data/silver/amostras_recortadas.json', 'r') as f:
    amostras_recortadas = json.load(f)

# for amostra in amostras_recortadas:
#     amostra['classe'] = dados.loc[dados['codigosai'] == amostra['codigo'], 'classificacao'].values[0]


# with open('../data/silver/amostras_recortadas.json', 'w') as f:
#     json.dump(amostras_recortadas, f, ensure_ascii=False, indent=4)

In [5]:
# Cria as features:
path_imagens = '../data/silver/amostras_recortadas_quadrado/'

labels = []
images = []
for amostra in amostras_recortadas:
    codigo = amostra['codigo']
    classe = amostra['classe']
    qtd_images = amostra['atributos']['quadrados']['qtd']
    for i in range(1, qtd_images):
        path_imagem = f"{path_imagens}AB_{codigo}_{i}.png"
        img = cv2.imread(path_imagem)
        images.append(img)
        labels.append(classe)

In [6]:
images = np.array(images)
images.shape

(6830, 256, 256, 3)

In [7]:
# Inicializar e treinar o PCANet

pcanet = PCANet(
    image_shape=256,
    filter_shape_l1=2, step_shape_l1=1, n_l1_output=3,  # parameters for the 1st layer
    filter_shape_l2=2, step_shape_l2=1, n_l2_output=3,  # parameters for the 2nd layer
    filter_shape_pooling=2, step_shape_pooling=2        # parameters for the pooling layer
)

In [8]:
pcanet.validate_structure()

In [8]:
images.shape

(6830, 256, 256, 3)

In [10]:
pcanet.fit(images)

# Extrair características usando PCANet
X = pcanet.transform(images)

KeyboardInterrupt: 

In [12]:
import numpy as np
import chainer.functions as F
from scipy.signal import convolve2d

# Entrada
input = np.random.rand(1, 3, 5, 5).astype(np.float32)
print(input)
print(input.shape)
kernel = np.random.rand(1, 3, 3, 3).astype(np.float32)

# Chainer
chainer_result = F.convolution_2d(input, kernel).data[0, 0]
print(chainer_result)
print(chainer_result.shape)

# SciPy
# SciPy: Convolução manual para 3 canais
input_scipy = input[0]  # Formato SciPy: (canais, altura, largura)
kernel_scipy = kernel[0]  # Formato SciPy: (canais, altura_kernel, largura_kernel)
output_scipy = np.zeros((3, 3))  # Saída para modo 'valid'

# Somar as convoluções de cada canal
for c in range(3):
    output_scipy += convolve2d(
        input_scipy[c], 
        kernel_scipy[c], 
        mode='valid'
    )

print(output_scipy)
print(output_scipy.shape)

# Verificar se os resultados são iguais
# np.testing.assert_allclose(chainer_result, scipy_result, rtol=1e-5)


[[[[0.16894887 0.10316014 0.07658041 0.12631202 0.80739164]
   [0.37392583 0.12090553 0.40288183 0.5457299  0.65873814]
   [0.9878692  0.17159341 0.5880632  0.29122785 0.6366386 ]
   [0.9554976  0.07179936 0.42247915 0.73408    0.39098385]
   [0.3662496  0.5959116  0.2692187  0.40942127 0.8453679 ]]

  [[0.02533662 0.7028205  0.02029976 0.43402275 0.89159524]
   [0.03879824 0.08642003 0.68688947 0.26499885 0.41300184]
   [0.5694126  0.7145486  0.66927475 0.32021284 0.39239362]
   [0.67229694 0.74395883 0.56297404 0.8461225  0.21803984]
   [0.17392471 0.7000573  0.64240986 0.7561727  0.36254984]]

  [[0.95593303 0.80035824 0.98404276 0.567369   0.46510485]
   [0.7418484  0.06555729 0.34187332 0.20232259 0.31035736]
   [0.18422504 0.72640675 0.10897689 0.17278416 0.53553945]
   [0.616962   0.22576906 0.03276759 0.90481883 0.9334807 ]
   [0.7488003  0.35940468 0.8277337  0.2112876  0.23610173]]]]
(1, 3, 5, 5)
[[5.7685146 5.09095   7.108759 ]
 [6.72532   6.7828593 7.3455358]
 [7.728493  7.

In [22]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

X = np.fromfile('../data/silver/PCANetFeatures.dat')
X = X.reshape(6830, X.size // 6830)
print(X.shape)

with open('../data/silver/amostras_recortadas.json', 'r') as f:
    amostras_recortadas = json.load(f)

labels = []
for amostra in amostras_recortadas:
    codigo = amostra['codigo']
    classe = amostra['classe']
    qtd_images = amostra['atributos']['quadrados']['qtd']
    for i in range(1, qtd_images):
        labels.append(classe)

labels = np.array(labels)

enc = LabelEncoder()
labels = enc.fit_transform(labels)

# Dividir os dados em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42)

# Treinar o modelo SVM
svm = SVC(kernel='rbf')
svm.fit(X_train, y_train)

# Avaliar o modelo
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo: {accuracy:.2f}")

(6830, 2560)
Acurácia do modelo: 0.20


In [ ]:
X_test[0]


(2560,)